In [5]:
import json
import time
import random

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


DOC_URLS = [
    "https://indiankanoon.org/doc/124827263/",
    "https://indiankanoon.org/doc/190336/",
    "https://indiankanoon.org/doc/1906475/",
]

OUTPUT_FILE = "indian_kanoon_precedents.json"


def get_source_name(driver):
    """
    Try to get current judgment/case name.
    Falls back to browser title.
    """
    selectors = [
        "h1",
        "h2",
        ".doc_title",
        ".judgments h2",
        ".judgments h1",
    ]

    for selector in selectors:
        elems = driver.find_elements(By.CSS_SELECTOR, selector)
        for elem in elems:
            text = elem.text.strip()
            if text:
                return text

    title = driver.title.strip()
    title = title.replace(" | Indian Kanoon", "").strip()
    return title


chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 25)

final_json = []

for i, doc_url in enumerate(DOC_URLS, start=1):
    print(f"\nOpening {i}/{len(DOC_URLS)}: {doc_url}")

    driver.get(doc_url)
    time.sleep(random.uniform(2, 4))

    try:
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "#citeselect"))
        )
    except TimeoutException:
        print("Dropdown not found.")
        print("If captcha/challenge is visible, solve it manually in Chrome.")
        input("After page loads fully, press Enter here...")

    source_name = get_source_name(driver)

    precedents = []

    options = driver.find_elements(By.CSS_SELECTOR, "#citeselect option[value]")

    for opt in options:
        ref_doc_id = opt.get_attribute("value")
        ref_name = opt.get_attribute("textContent").strip()

        if not ref_doc_id:
            continue

        if "select precedent" in ref_name.lower():
            continue

        precedents.append({
            "name": ref_name,
            "doc_id": ref_doc_id,
            "url": f"https://indiankanoon.org/doc/{ref_doc_id}/"
        })

    final_json.append({
        "source_name": source_name,
        "source_url": doc_url,
        "precedent_count": len(precedents),
        "precedents": precedents
    })

    print(f"Source name: {source_name}")
    print(f"Found {len(precedents)} precedents")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(final_json, f, indent=2, ensure_ascii=False)

    time.sleep(random.uniform(3, 7))


print(f"\nSaved JSON to {OUTPUT_FILE}")
print("Done.")


Opening 1/3: https://indiankanoon.org/doc/124827263/
Source name: Indian Kanoon - Search engine for Indian Law
Found 2 precedents

Opening 2/3: https://indiankanoon.org/doc/190336/
Source name: Indian Kanoon - Search engine for Indian Law
Found 9 precedents

Opening 3/3: https://indiankanoon.org/doc/1906475/
Source name: Indian Kanoon - Search engine for Indian Law
Found 2 precedents

Saved JSON to indian_kanoon_precedents.json
Done.
